# Test Stage 6 — Does the data reach memory intact?

**What this stage checks:** the transport, and *only* the transport. Packing, buffering, the
write into memory, and the layout Python reads back — with no signal processing involved at
all.

That separation is the whole point. Debugging data movement while a real signal is flowing is
miserable, because a transport fault and a signal-processing fault look identical: numbers
that are not what you expected. Here the data is a **counting sequence**, so the right answer
is known exactly and every kind of failure has its own signature.

| What you see | What it means |
|---|---|
| An unbroken count | Every sample arrived, in order, undamaged |
| Gaps in the count | Samples were **dropped** — the buffer overflowed |
| Repeated values | Something stalled |
| `I + Q ≠ 0xFFFFFFFF` | The two halves of each 64-bit word are swapped or corrupted |
| Right count, wrong field | I and Q are the wrong way round in memory |

The pattern is `I = n`, `Q = NOT n`, so `I + Q` must equal `0xFFFFFFFF` for every sample —
an invariant that catches corruption a plain "both count up" pattern would miss.

---

## ⚠ Before anything else: the memory region

This block writes **directly into physical memory**, at whatever address software gives it.
Nothing stops it writing over Linux. Point it at the wrong place and the board will crash, or
worse, corrupt the filesystem.

The region must be one Linux has been told **not** to use. The next cell works out where that
is and refuses to go further if it cannot establish it safely.

## Setup

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
# =============================================================================
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer the bus bridge cannot acknowledge; it locks
#  the bus and reboots the board. Do not "optimise" these loops into slices.
#
#  NOTE ON NUMPY: the board's numpy is old. Use np.polyfit rather than
#  np.linalg.lstsq(..., rcond=None), which fails with "a float is required".
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0]en [1]soft-rst [2]test [3]tap [4]arm [5]wrap [6]fir-byp [7]dma-clr [8]axi-port(0=axi0)
R_LO_FINC    = 0x04
R_DEC_LOG2   = 0x08
R_SRC_SEL    = 0x0C
R_TAP_SEL    = 0x10   # 0=mixer 1=decimator 2=corrector
R_TAP_SHIFT  = 0x14
R_PEEK_I     = 0x18
R_PEEK_Q     = 0x1C
R_SAMPLE_CNT = 0x20
R_STATUS     = 0x24
R_MEM_START  = 0x28
R_MEM_STOP   = 0x2C
R_MEM_CUR    = 0x30
R_DMA_COUNT  = 0x34
R_STATUS2    = 0x38   # [0]mem overflow [1]filter overrun [2]filter bypassed [3]write error


def s32(v):
    """Reinterpret a 32-bit register value as signed."""
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val): self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):      return struct.unpack('<I', self.m[off:off+4])[0]

    @staticmethod
    def finc(f_lo):
        """f_out = f_clk * finc / 2**32, so finc = f_out/f_clk * 2**32."""
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    def setup(self, f_lo, dec_log2, src=0, test=False, tap_en=False,
              tap_sel=0, tap_shift=10, fir_bypass=False,
              arm=False, wrap=True, mem_start=None, mem_stop=None, axi_port=0):
        """Configure and start. ORDER MATTERS: the decimator holds a running
        total that assumes one fixed decimation factor, so always
        disable -> write settings -> reset -> enable."""
        self.w(R_CTRL, 0)                                  # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))               # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        if mem_start is not None: self.w(R_MEM_START, mem_start)
        if mem_stop  is not None: self.w(R_MEM_STOP,  mem_stop)
        ctrl = (1 |
                (4  if test       else 0) |
                (8  if tap_en     else 0) |
                (16 if arm        else 0) |
                (32 if wrap       else 0) |
                (64  if fir_bypass else 0) |
                (256 if axi_port    else 0))    # CTRL[8]: 0 = axi0 (default), 1 = axi1
                                              # (axi1 drops Q halves - Step 3)
        self.w(R_CTRL, ctrl | 2 | 128)                     # 3. enable + reset + dma clear
        self.w(R_CTRL, ctrl)                               # 4. running

    def disable(self):  self.w(R_CTRL, 0)
    def peek(self):     return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))
    def count(self):    return self.r(R_SAMPLE_CNT)

    def status(self):
        s = self.r(R_STATUS2)
        return dict(mem_overflow=bool(s & 1),
                    filter_overrun=bool(s & 2),
                    filter_bypassed=bool(s & 4),
                    write_error=bool(s & 8))


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
st = acq.status()
print('status: memory overflow=%s  filter overrun=%s  filter bypassed=%s'
      % (st['mem_overflow'], st['filter_overrun'], st['filter_bypassed']))

---
## Step 1 — Find a safe region of memory

### What this is for

The memory writer puts samples **straight into the board's RAM**, at whatever physical address
software gives it. It goes through the processor's high-speed memory port and knows nothing
about Linux: no permissions, no allocator, no bounds. Point it at an address Linux is using and
it will overwrite whatever is there — a running program, a disk cache, the kernel itself. The
symptoms range from a crash to silent filesystem corruption.

So before arming it, we need a region of RAM that Linux has been told **not to touch**.

### Two ways that region normally exists

| | How it looks |
|---|---|
| **A limit on the kernel command line** — `mem=480M` | Linux uses the bottom 480 MB of a 512 MB board and never looks above it |
| **A reservation in the device tree** | Linux is told about all the memory but marked a block as off limits |

The first is what Red Pitaya's own documentation describes. The second is what some newer
images do instead. Either is fine, and this step accepts both — it works out what Linux
**actually claims** from `/proc/iomem` and compares it with how much RAM the board really has.
Whatever is left over, if anything, is ours.

### It does not affect the FPGA build

This is a setting on the operating system, not on the logic. Nothing is rebuilt, no bitstream
changes, and the addresses are ordinary register writes at run time. If you change it you
reboot the board; you do not go near Vivado.

**It refuses to continue if it cannot establish a region.** Guessing an address here is not an
acceptable risk.


In [ ]:
import re

MEM_OK = False
MEM_START = MEM_STOP = None


def be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            continue
    return None


def dt_dir(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        p = os.path.join(root, path)
        if os.path.isdir(p):
            return p
    return None


# --- 1. blocks the kernel has been TOLD to set aside -----------------------
#  This is the best kind of answer. A reserved-memory node is a promise: the
#  kernel knows the block exists and has been instructed not to allocate it.
print('Reserved for hardware, declared in the device tree:')
regions = []
resv = dt_dir('reserved-memory')
if resv:
    for name in sorted(os.listdir(resv)):
        if not os.path.isdir(os.path.join(resv, name)):
            continue
        blob = dt_read(os.path.join('reserved-memory', name, 'reg'))
        if blob and len(blob) >= 8:
            a, n = be32(blob, 0), be32(blob, 1)
            print('   %-24s 0x%08X .. 0x%08X   %6.1f MB'
                  % (name, a, a + n - 1, n / 1024.0**2))
            regions.append((name, a, n))
        else:
            print('   %-24s (dynamically placed, cannot be used here)' % name)
if not regions:
    print('   none')
print('')

# --- 2. memory that was never declared at all ------------------------------
#  The device tree's memory node says how much Linux is GIVEN, not how much is
#  fitted. Red Pitaya images routinely declare less than is present, which
#  leaves the top of memory unreachable by Linux - and therefore ours.
blob = dt_read('memory/reg') or dt_read('memory@0/reg')
declared_end = None
if blob and len(blob) >= 8:
    base, total = be32(blob, 0), be32(blob, 1)
    declared_end = base + total
    print('Declared to Linux : 0x%08X .. 0x%08X   (%.0f MB)'
          % (base, declared_end - 1, total / 1024.0**2))
    if declared_end < 512 * 1024**2:
        print('This board has 512 MB fitted, so 0x%08X .. 0x1FFFFFFF (%.0f MB) is'
              % (declared_end, (512 * 1024**2 - declared_end) / 1024.0**2))
        print('present but never declared - Linux cannot use what it was not told about.')
print('')

# --- 3. pick one -----------------------------------------------------------
# ---------------------------------------------------------------------------
#  RANKING, AND WHY IT IS THIS WAY ROUND
#
#  MEMORY THE KERNEL WAS NEVER TOLD ABOUT COMES FIRST.
#
#  A reserved-memory block sounds like the better answer - the kernel knows it
#  exists and has been told not to allocate it. But "not allocated" is not the
#  same as "not mapped": a block inside the declared memory is still part of the
#  kernel's linear map, so the processor holds a CACHEABLE view of it as well as
#  the uncached one /dev/mem gives us.
#
#  That matters because the logic writes to memory through a port that bypasses
#  the processor's caches entirely. Two views of the same addresses, one cached
#  and one not, is precisely the situation where a write through one is not
#  visible through the other. On 2026-07-29 the writer reported 57129 accepted
#  writes into labuf and the processor read back the fill pattern, unchanged.
#
#  Memory above the declared end has no such problem. The kernel never saw it,
#  so there is no second view of it and nothing to be stale. That is also why
#  Red Pitaya's own images shrink the memory node rather than only reserving:
#  the top of RAM becomes genuinely unreachable, not merely spoken for.
# ---------------------------------------------------------------------------
candidates = []
if declared_end is not None and declared_end < 512 * 1024**2:
    gap = 512 * 1024**2 - declared_end
    candidates.append((1 << 40, declared_end, gap,
                       'above the declared memory - no cached view of it exists'))
for name, a, n in regions:
    if n >= 1024**2:
        candidates.append((n, a, n,
                           'reserved-memory: %s (inside the kernel map)' % name))

if candidates:
    candidates.sort(reverse=True)
    _, a, n, why = candidates[0]
    # -----------------------------------------------------------------------
    #  MEM_STOP IS THE LAST ADDRESS, NOT ONE PAST THE END.
    #
    #  From red_pitaya_acq_dma.v:
    #      input [AW-1:0] set_stop_i ,  //!< last address of the region
    #
    #  So the writer is entitled to write the 8-byte word AT MEM_STOP, and its
    #  address counter then reads MEM_STOP + 8. Setting MEM_STOP to "one past
    #  the end" therefore lets it write 8 bytes further than intended - which is
    #  exactly what happened on 2026-07-29, harmlessly, and looked like a fault.
    #
    #  Computed as start + size - 8 so the very last byte the writer can touch
    #  is MEM_START + size - 1, and checked against the region below.
    # -----------------------------------------------------------------------
    MEM_START = (a + 0xFFF) & ~0xFFF                     # page aligned
    size      = min(0x00800000, (a + n) - MEM_START)     # 8 MB is ample
    size      = size & ~7                                # whole 64-bit words
    MEM_STOP  = MEM_START + size - 8                     # LAST address, inclusive
    MEM_OK    = True

    #  Belt and braces: the furthest byte the writer can reach must still be
    #  inside the region we chose.
    last_byte = MEM_STOP + 7
    if last_byte > a + n - 1:
        MEM_OK = False
        print('INTERNAL ERROR: the computed region overruns %s. Not proceeding.' % why)
    print('Using: %s' % why)
    print('   0x%08X .. 0x%08X inclusive   %.1f MB, room for %d samples'
          % (MEM_START, MEM_STOP + 7, (MEM_STOP + 8 - MEM_START) / 1024.0**2,
             (MEM_STOP + 8 - MEM_START) // 8))
    if 'labuf' in why:
        print('')
        print('   labuf is Red Pitaya\'s logic-analyser buffer. It is free unless you')
        print('   run their logic analyser application - do not run both at once.')
else:
    print('*** NO USABLE REGION ***')
    print('')
    print('   No reserved block big enough, and no memory beyond what was declared.')
    print('   Run  sudo python3 show_memory_map.py  for the full picture, and see')
    print('   the next cell. Do not guess an address.')


# ---------------------------------------------------------------------------
#  BEFORE TRUSTING IT: PROBE THE REGION
#
#  A device tree entry is a statement of intent. This checks it empirically:
#  write a signature into the region from Linux, wait, and read it back. If
#  anything else is living there, it will very likely have written over part of
#  the pattern while we waited.
#
#  Finding it intact is strong evidence that nothing is using the region right
#  now. It is NOT a guarantee that nothing will start. It is a seat belt.
# ---------------------------------------------------------------------------
def probe_region(start, nbytes=0x100000, dwell=2.0):
    """Returns the number of 32-bit words that changed while we waited."""
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=start)
    try:
        sig = (np.arange(nbytes // 4, dtype='<u4') ^ np.uint32(0xA5A5A5A5))
        mm[0:nbytes] = sig.tobytes()
        time.sleep(dwell)
        back = np.frombuffer(mm[0:nbytes], dtype='<u4')
        return int(np.count_nonzero(back != sig))
    finally:
        mm.close()
        os.close(fd)


if MEM_OK:
    print('')
    print('probing it before trusting it (2 seconds at each end)...')
    bad_lo = probe_region(MEM_START)
    bad_hi = probe_region((MEM_STOP + 8 - 0x100000) & ~0xFFF)
    print('   words disturbed at the start : %d' % bad_lo)
    print('   words disturbed at the end   : %d' % bad_hi)
    if bad_lo or bad_hi:
        MEM_OK = False
        print('')
        print('*** SOMETHING ELSE IS USING THIS MEMORY ***')
        print('   A pattern written from Linux did not survive two seconds, so this')
        print('   region is in use despite being declared reserved. Do not arm the')
        print('   writer. Try another region from the list above.')
    else:
        print('   Untouched. Nothing is using it.')

# ---------------------------------------------------------------------------
#  NOW POISON IT
#
#  The probe leaves its own pattern lying in the region, and on 2026-07-29 that
#  pattern was read back by Step 3 and mistaken for acquisition data - it looked
#  like every sample was corrupt, when in truth the writer had never written
#  there at all.
#
#  So fill the whole region with a value that could not possibly be data.
#  0xDEADBEEF never appears in a counting sequence and never appears in a
#  measurement. Afterwards, any word still reading 0xDEADBEEF is a word the
#  hardware DID NOT WRITE, which turns "the data looks wrong" into "the data
#  never arrived" - a completely different fault with a completely different
#  cause.
#
#  This also gives Step 3 an exact high-water mark: where the poison begins is
#  where the writer stopped.
# ---------------------------------------------------------------------------
POISON = 0xDEADBEEF

if MEM_OK:
    nbytes = MEM_STOP + 8 - MEM_START
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, nbytes, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)
    print('   filled %.1f MB with 0x%08X, so anything the writer does not write'
          % (nbytes / 1024.0**2, POISON))
    print('   will be obvious rather than mistakable for data.')

print('')
print('STEP 1:', 'PASS' if MEM_OK else 'STOP - do not continue without a safe region')


### If Step 1 found no free memory

**Stop and look before changing anything.** On the board:

```
sudo python3 show_memory_map.py
```

That is read-only — it mounts nothing, writes nothing and reboots nothing. It prints how much
RAM is fitted, how much Linux has claimed, whether anything is already reserved, and where this
particular image keeps its boot settings.

### Why "just edit uEnv.txt" may not apply

Red Pitaya ships several different images and they do not all boot the same way. On this board:

```
root@rp-f007f7:~# mount | grep mmcblk0p1
/dev/mmcblk0p1 on /boot type vfat (ro,relatime,...)

root@rp-f007f7:~# ls /boot
bin  boot.bin  devicetree.dtb  dtraw.dts  fpga  include  lib  rp_sdk
sbin  u-boot.scr  uImage  version.txt  www
```

Three things to notice:

| | |
|---|---|
| **There is no `uEnv.txt`** | so there is no text file of boot arguments to edit |
| **`u-boot.scr` and `devicetree.dtb` are compiled** | a text editor cannot change them meaningfully. They need `mkimage` and `dtc` |
| **The partition is mounted `ro`** | read-only. Nothing can be written to it without remounting first |

So the standard advice does not fit this image, and following it anyway — remounting the boot
partition read-write and editing compiled files by guesswork — is a good way to end up with a
board that does not boot. `dtraw.dts` is the device tree's source and is worth reading, but
recompiling it is a real change to how the board starts.

### What to do instead, in order of preference

**1. Find out whether a region already exists.** Many Red Pitaya images reserve a block for
exactly this purpose, because their own deep-memory acquisition needs one. `show_memory_map.py`
looks in all the places it can be declared. If it finds one, **nothing needs changing** — set
`MEM_START` and `MEM_STOP` to sit inside it and carry on.

**2. Ask whoever set this board up.** A region may exist and simply not be declared where a
script can see it. This is a two-minute question that avoids an afternoon of risk.

**3. Use a Red Pitaya image that reserves memory.** Swapping the SD card for a stock image with
a reserved buffer is safer than modifying this one, and completely reversible — keep the
original card.

**4. Rebuild the boot files.** Possible, and the last resort. It changes how the board starts,
so image the SD card first (see `../BACKUP.md`), and expect to need a card reader to recover.

### Meanwhile, the rest of the project is unaffected

Stages 1 to 5 need none of this. They read registers and read memory, and they have already
established that the whole signal path works. Stage 6 tests the route into memory and Stage 7
needs Stage 6 — but nothing that has been verified so far depends on solving this.

### Setting the addresses by hand

If you already know a reserved region, skip everything above and set the values directly:

```python
MEM_START = 0x1E100000
MEM_STOP  = 0x1E900000
MEM_OK    = True
```

**Only if you are certain.** A wrong address here does not produce a wrong answer, it corrupts
whatever is living there. It must also be a multiple of 4096, because the writer moves data in
bursts and expects to start on a page boundary.

---

## How risky is this, really?

Three things now stand between a wrong address and a damaged system, and they are independent
of each other:

| Guard | What it catches |
|---|---|
| **The gap check** | Refuses unless Linux demonstrably stops below the top of RAM |
| **The probe** | Writes a signature into the region from Linux, waits two seconds, reads it back. If anything else is living there the pattern will not survive |
| **The 4 kB trial** | The first time the writer is ever armed it gets a window half a page wide. If the address is wrong, that is the size of the damage — and the step checks the writer stayed inside its window before letting it near the full region |

Be clear about what the probe proves. Finding the pattern intact is strong evidence that nothing
is using the region *now*. It is **not** a guarantee that nothing will start using it in a
minute. Only a reservation gives that, which is why the probe is a seat belt rather than a
reason to skip it.

The worst realistic outcome, with all three in place, is a corrupted 4 kB somewhere harmless and
a reboot — and an SD card image makes even that a non-event. See `../BACKUP.md`.


### If the data does not appear where it was written

The most likely reason, and the one to try first, is **two views of the same memory**.

The logic writes through a port that bypasses the processor's caches. If the region is inside
the memory the kernel was told about, the processor also holds a *cacheable* view of those same
addresses — and a write through one view is not reliably visible through the other. The symptom
is exact and distinctive:

| | |
|---|---|
| The writer's address counter | advances by exactly 8 bytes per sample |
| The overflow flag | stays clear |
| Reading the region back | returns whatever was there before, unchanged |

Everything says the writes happened. They did — the processor is just looking at a stale copy.

**The fix is to use memory the kernel was never told about.** On this board the device tree
declares 480 MB of the 512 MB fitted, so `0x1E000000 .. 0x1FFFFFFF` exists and Linux has no view
of it at all — cached or otherwise. Step 1 now prefers that region for exactly this reason.

To force it by hand, put this in a cell of its own and run Steps 2 and 3 again:

```python
MEM_START = 0x1E000000
MEM_STOP  = 0x1E7FFFF8      # last address, inclusive
MEM_OK    = True
```

If the data appears there and not in `labuf`, that confirms it: the transport works, and the
difference is only which addresses the processor can see honestly.


---
## Step 2 — Capture a counting sequence

Switch on test-pattern mode, point the writer at the region, arm it, wait, and stop.

`wrap = False` makes this a **single capture**: the writer fills the region once and stops,
rather than continuously overwriting the oldest data. That is what we want for a check like
this — a fixed, finite record with a definite beginning.

In [ ]:
assert MEM_OK, 'Step 1 did not establish a safe memory region'

DEC = 10                       # R = 1024 -> 122 kSample/s
R   = 1 << DEC
N_WANT = 20000                 # samples to capture

# ---------------------------------------------------------------------------
#  TOE IN THE WATER FIRST
#
#  The very first time the writer is armed is the moment a wrong address does
#  its damage. So arm it once over a deliberately tiny window - 4 kB, half a
#  page - and check it behaved before letting it near the full region. If the
#  address is wrong, 4 kB of damage is recoverable; 8 MB through the middle of a
#  running filesystem is not.
# ---------------------------------------------------------------------------
TRIAL_BYTES = 0x1000                       # 4 kB: half a page
TRIAL_STOP  = MEM_START + TRIAL_BYTES - 8  # LAST address, matching the hardware

print('first, a %d byte trial capture...' % TRIAL_BYTES)
print('   window 0x%08X .. 0x%08X inclusive, %d samples'
      % (MEM_START, TRIAL_STOP + 7, TRIAL_BYTES // 8))
#  wrap=True on purpose. With wrap=False the writer stops when it reaches the
#  end of its window and latches that it has finished - a state the full capture
#  then has to clear. Wrapping is just as bounded (that is what wrapping means)
#  and leaves nothing behind, so the trial cannot be the reason a later capture
#  fails. It also exercises the wrap logic, which the real capture does not.
#  Which of the processor's two memory ports to write through. Everything in
#  the logic is identical either way, so if one works and the other does not,
#  the difference is the port. CTRL[8] chooses; no rebuild needed.
# Which of the two memory ports to write through. axi0 is the default and the
# RIGHT one on this board: the boot software leaves axi1's processor-side port
# (HP1) with a channel in 32-bit mode, which silently discards the upper half
# of every 64-bit sample - I lands, Q never does. Confirmed on hardware
# 2026-07-30; see CHANGELOG.md. axi1 stays selectable for diagnostics only.
AXI_PORT = 0          # 0 = axi0 (use this), 1 = axi1 (drops Q halves - see above)

TRIAL_WRAP = True
acq.setup(f_lo=1e6, dec_log2=DEC, test=True, arm=True, wrap=TRIAL_WRAP,
          mem_start=MEM_START, mem_stop=TRIAL_STOP, axi_port=AXI_PORT)
time.sleep(0.2)
acq.w(R_CTRL, acq.r(R_CTRL) & ~16)

trial_n   = acq.r(R_DMA_COUNT)
trial_cur = acq.r(R_MEM_CUR)
trial_ovf = acq.status()['mem_overflow']
print('   samples handed to the writer : %d' % trial_n)
print('   window holds                 : %d' % (TRIAL_BYTES // 8))
print('   address counter now          : 0x%08X' % trial_cur)
#  Do not label this in advance - say what it means for the settings actually
#  used. With wrap=True the writer goes round its window for ever and can never
#  overflow, so False is correct and True would be a real fault. With
#  wrap=False it fills the window, refuses the rest, and True is correct.
#  The label was hard-coded as "EXPECTED" while the trial used wrap=False, and
#  became wrong the moment the trial was changed to wrap.
if trial_ovf:
    note = 'a fault: wrapping cannot overflow' if TRIAL_WRAP else 'expected: the window is deliberately too small'
else:
    note = 'correct: a wrapping writer never runs out of room' if TRIAL_WRAP else 'unexpected: the window should have filled'
print('   overflow flag                : %-5s <- %s' % (trial_ovf, note))
print('')

# ---------------------------------------------------------------------------
#  WHAT PASSES AND WHAT DOES NOT, AND WHY
#
#  OVERFLOW IS EXPECTED HERE and is not a fault. The chain keeps producing
#  samples at 122000 a second while this runs; the window holds 512. Everything
#  after the first 512 is correctly refused and the flag correctly says so.
#  Deliberately overflowing a small window is a large part of what this trial
#  is for.
#
#  THE ADDRESS IS THE THING TO CHECK. The counter points at where the NEXT word
#  would go, so after filling the window it legitimately reads TRIAL_STOP + 8.
#  Anything beyond that means the writer ignored its bounds, which is the one
#  failure that could damage something.
# ---------------------------------------------------------------------------
if trial_ovf and TRIAL_WRAP:
    raise SystemExit('The overflow flag is set while wrapping, which should be '
                     'impossible - a wrapping writer always has somewhere to '
                     'put the next sample. Do not continue.')
if trial_n == 0:
    raise SystemExit('The writer produced nothing. Do not continue - check the '
                     'chain is enabled and the test pattern is on.')

lo, hi = MEM_START, TRIAL_STOP + 8
if trial_cur < lo or trial_cur > hi:
    raise SystemExit(
        'THE WRITER IGNORED ITS BOUNDS.\n'
        '  window     : 0x%08X .. 0x%08X\n'
        '  counter at : 0x%08X  (%d bytes outside)\n'
        'Do not arm it again. This is a hardware fault, not a test artefact.'
        % (lo, hi, trial_cur, trial_cur - hi))

print('   The writer stopped where it was told: the counter is at 0x%08X,' % trial_cur)
print('   which is the word after the last one it was allowed to write.')
print('   Safe to use the full region.')
print('')

acq.setup(f_lo=1e6, dec_log2=DEC, test=True, arm=True, wrap=False,
          mem_start=MEM_START, mem_stop=MEM_STOP, axi_port=AXI_PORT)

expect_s = N_WANT * R / FS_CLK
print('capturing %d samples at %.1f kSample/s  (~%.2f s)'
      % (N_WANT, FS_CLK / R / 1e3, expect_s))
time.sleep(expect_s + 0.3)

acq.w(R_CTRL, acq.r(R_CTRL) & ~16)          # disarm, leaving the chain running

n_written = acq.r(R_DMA_COUNT)
cur       = acq.r(R_MEM_CUR)
st        = acq.status()

# ---------------------------------------------------------------------------
#  DID IT ACTUALLY WRITE?
#
#  DMA_COUNT counts samples handed TO the writer, not samples that reached
#  memory. The two are different, and on 2026-07-29 they disagreed: the counter
#  said 57061 while the region still held the pattern Step 1 had probed it with.
#  Reading the address counter is what tells them apart - it only moves when
#  memory accepts a write.
# ---------------------------------------------------------------------------
advanced = cur - MEM_START
expected = int(n_written) * 8
print('address counter advanced : %d bytes  (expected about %d)'
      % (advanced, expected))

if advanced == 0:
    print('')
    print('*** THE WRITER HANDED OVER %d SAMPLES AND MOVED NOWHERE ***' % n_written)
    print('')
    print('   Nothing reached memory. The samples were produced and counted, so')
    print('   the chain is running - what failed is the write itself. Check:')
    print('     1. the arm bit is set:      CTRL reads 0x%02X' % acq.r(R_CTRL))
    print('     2. the region was accepted: MEM_START reads 0x%08X, MEM_STOP 0x%08X'
          % (acq.r(R_MEM_START), acq.r(R_MEM_STOP)))
    print('     3. whether a previous capture left the writer finished - the')
    print('        trial above now wraps rather than stopping, precisely so it')
    print('        cannot do that')
    raise SystemExit('stopping: nothing reached memory')

if advanced < expected // 2:
    print('   Much less than expected - memory accepted only some of it.')

print()
print('samples handed to the writer : %d' % n_written)
print('current write address        : 0x%08X  (%d bytes into the region)'
      % (cur, cur - MEM_START))
print('memory overflow flag         : %s' % st['mem_overflow'])
# The annotation must match the value: printing 'memory REFUSED a write'
# next to False (as this line did until 2026-07-30) reads like an error
# when everything is fine.
print('WRITE ERROR flag             : %s%s'
      % (st['write_error'],
         '   <- memory REFUSED a write' if st['write_error'] else '   <- every write succeeded'))
print('memory port used             : axi%d' % AXI_PORT)

# ---------------------------------------------------------------------------
#  The write-error bit is the one that separates "accepted" from "succeeded".
#  Until 2026-07-29 it was wired into the logic and never brought out, which is
#  why a transfer that reported perfect success could leave memory untouched.
# ---------------------------------------------------------------------------
if st['write_error']:
    print('')
    print('*** MEMORY REFUSED THE WRITES ***')
    print('   The writes were issued and answered with an error rather than an')
    print('   acknowledgement, so nothing was stored. This is not the logic - it')
    print('   is the port or the address. Try the other port: set AXI_PORT = %d'
          % (1 - AXI_PORT))
    print('   above and run this cell again. No rebuild needed.')
print()
if st['mem_overflow']:
    print('OVERFLOW - samples were lost. The record would still LOOK continuous,')
    print('but time has jumped, so every frequency derived from it would be wrong.')
# The verdict must test all three ways this step can fail, not just two.
# Until 2026-07-30 it ignored write_error, so a run in which memory refused
# every single write still printed PASS - a verdict measuring acceptance,
# not success. See CHANGELOG.md.
print('STEP 2:', 'PASS' if (n_written > 1000
                            and not st['mem_overflow']
                            and not st['write_error']) else 'FAIL')

---
## Step 3 — Read it back and check every sample

Now read the region and verify the count.

**A note on how this read differs from register reads.** Everywhere else these notebooks
insist on single 32-bit accesses, because the register space goes through a bridge that cannot
handle bursts. **This is different.** Ordinary memory is reached by the processor's normal
memory path, not that bridge, so bulk reads are both safe and fast. A bulk read of thousands
of samples takes milliseconds.

In [ ]:
n_read = min(int(n_written), (MEM_STOP + 8 - MEM_START) // 8)
n_read = min(n_read, 200000)

fd  = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
pagesize = 4096
off_aligned = (MEM_START // pagesize) * pagesize
delta       = MEM_START - off_aligned
length      = ((delta + n_read * 8 + pagesize) // pagesize) * pagesize

mm  = mmap.mmap(fd, length, offset=off_aligned)
buf = mm[delta:delta + n_read * 8]           # bulk read: safe for ordinary memory
mm.close(); os.close(fd)

# Each 64-bit word is {Q, I}. The processor is little-endian, so read as signed
# 32-bit values the order is I, Q, I, Q, ...
raw = np.frombuffer(buf, dtype='<u4')
I   = raw[0::2].astype(np.int64)
Q   = raw[1::2].astype(np.int64)

print('read back %d samples' % len(I))
print()
print('  first four I : %s' % [hex(int(v)) for v in I[:4]])
print('  first four Q : %s' % [hex(int(v)) for v in Q[:4]])
print()

# ---------------------------------------------------------------------------
#  BEFORE CHECKING THE DATA: IS IT DATA?
#
#  Two patterns can be sitting here that are not acquisition output, and both
#  have fooled this test before. Recognise them by name rather than reporting
#  thousands of corrupt samples.
# ---------------------------------------------------------------------------
POISON = 0xDEADBEEF
untouched = int(np.count_nonzero(raw == POISON))
probe_sig = (np.arange(min(len(raw), 64), dtype='<u4')
             ^ np.uint32(0xA5A5A5A5)).astype(np.int64)
is_probe  = len(raw) >= 64 and np.array_equal(raw[:64].astype(np.int64), probe_sig)

if is_probe:
    print('*** THIS IS NOT ACQUISITION DATA ***')
    print('')
    print('  It is the pattern Step 1 wrote when it probed the region')
    print('  (arange XOR 0xA5A5A5A5). The memory writer never wrote here.')
    print('')
    print('  Look at what Step 2 printed for "current write address". If it is')
    print('  still MEM_START, nothing was written at all. If it moved, the data')
    print('  went somewhere other than where this is reading.')
    raise SystemExit('stopping: the region still holds the probe pattern')

if untouched == len(raw):
    print('*** NOTHING WAS WRITTEN ***')
    print('')
    print('  Every word still reads 0x%08X, the value Step 1 filled the region' % POISON)
    print('  with. The writer reported handing over %d samples, so it believes' % n_written)
    print('  it did something - but nothing reached this memory.')
    print('')
    print('  Check in this order:')
    print('    1. Step 2\'s "current write address" - did it move at all?')
    print('    2. the arm bit - CTRL[4] must be set, and setup() sets it')
    print('    3. that MEM_START here is the same address Step 2 armed with')
    raise SystemExit('stopping: nothing was written to this region')

if untouched:
    first_poison = int(np.argmax(raw == POISON)) // 2
    print('  %d of %d words are still 0x%08X, i.e. never written.'
          % (untouched, len(raw), POISON))
    print('  The writer got as far as sample %d and stopped.' % first_poison)
    print('  Checking only the %d samples that were actually written.' % first_poison)
    print('')
    I = I[:first_poison]
    Q = Q[:first_poison]
    if len(I) < 100:
        raise SystemExit('stopping: only %d samples were written - too few to '
                         'check. See Step 2.' % len(I))
print()

# --- check 1: I + Q == 0xFFFFFFFF for every sample -----------------------
inv = (I + Q) == 0xFFFFFFFF
print('check 1  I + Q == 0xFFFFFFFF for all samples : %s  (%d of %d)'
      % (inv.all(), inv.sum(), len(inv)))
if not inv.all():
    bad = np.where(~inv)[0][:5]
    print('         first failures at %s' % bad)
    print('         -> the two halves of the 64-bit word are swapped or corrupted')

# --- check 2: an unbroken count -----------------------------------------
step = np.diff(I)
step_mod = np.mod(step, 1 << 32)             # allow the counter to wrap
good = (step_mod == 1)
print('check 2  I increases by exactly 1 each sample : %s  (%d of %d gaps)'
      % (good.all(), good.sum(), len(good)))
if not good.all():
    bad = np.where(~good)[0][:5]
    print('         first breaks at sample %s' % bad)
    for b in bad[:3]:
        d = int(step_mod[b])
        kind = ('DROPPED %d samples' % (d - 1)) if d > 1 else \
               ('REPEATED value' if d == 0 else 'OUT OF ORDER')
        print('           at %d: step of %d  -> %s' % (b, d, kind))

ok = inv.all() and good.all()
print()
print('STEP 3:', 'PASS' if ok else 'FAIL')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
n_show = min(2000, len(I))
ax[0].plot(I[:n_show] - I[0], lw=1)
ax[0].set_xlabel('sample'); ax[0].set_ylabel('I - I[0]')
ax[0].set_title('Should be a perfectly straight line')
ax[0].grid(True, alpha=0.4)

ax[1].plot(np.mod(np.diff(I), 1 << 32), '.', ms=2)
ax[1].set_xlabel('sample'); ax[1].set_ylabel('step size')
ax[1].set_ylim(-0.5, 4.5)
ax[1].set_title('Every step should be exactly 1.\nAnything above = dropped, 0 = repeated')
ax[1].grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

---
## Step 4 — Deliberately break it, and check the flag notices

A safety net that has never been seen to work is not a safety net. This step forces the
buffer to overflow by pointing the writer at a region far too small for the number of samples,
and confirms the flag catches it.

**Passes if:** the overflow flag reads True afterwards, and clears again when reset.

**If it stays False:** the flag is not wired up, and a real overflow later would silently
corrupt your data while every other indicator looked healthy.

In [ ]:
tiny_start = MEM_START
tiny_stop  = MEM_START + 0x400          # 1 kB = 128 samples: fills almost instantly

acq.setup(f_lo=1e6, dec_log2=6,          # R = 64: the fastest rate available
          test=True, arm=True, wrap=False,
          mem_start=tiny_start, mem_stop=tiny_stop)
time.sleep(0.5)
acq.w(R_CTRL, acq.r(R_CTRL) & ~16)

st = acq.status()
print('after deliberately overfilling a 1 kB region:')
print('   overflow flag = %s   (want True)' % st['mem_overflow'])

# clear it and confirm it goes away
acq.setup(f_lo=1e6, dec_log2=DEC, mem_start=MEM_START, mem_stop=MEM_STOP)
time.sleep(0.1)
st2 = acq.status()
print('   after clearing = %s   (want False)' % st2['mem_overflow'])
print()
ok = st['mem_overflow'] and not st2['mem_overflow']
print('STEP 4:', 'PASS' if ok else 'FAIL - the overflow flag is not reliable')

---
## Stage 6 summary

| Step | What it proves |
|---|---|
| 1 | A region of memory exists that Linux is not using |
| 2 | The writer fills it, and reports how much it wrote |
| 3 | Every sample arrives, in order, undamaged, with I and Q the right way round |
| 4 | The overflow flag genuinely detects lost samples |

Step 4 is the one people skip. Lost samples are the most dangerous failure this instrument
can have, because the record still looks like a continuous measurement — time has simply
jumped. A flag that works is what turns that from a silent corruption into a visible one.

**Before moving on:**

In [ ]:
acq.disable()
print('acquisition disabled')